In [ ]:
import requests
import csv

def get_gene_details(gene_name, genome):
    """
    Fetch gene details including chromosome and TSS position.
    """
    search_url = f"https://api.genome.ucsc.edu/search?search={gene_name}&genome={genome}"
    response = requests.get(search_url)
    if response.ok:
        data = response.json()
        if 'positionMatches' in data and data['positionMatches'] and data['positionMatches'][0]['matches']:
            first_match = data['positionMatches'][0]['matches'][0]
            position = first_match['position']
            try:
                chrom, positions = position.split(':')
                start, end = positions.split('-')
                return chrom, int(start)
            except ValueError:
                print(f"Invalid format for position data: {position}")
                return None, None
        else:
            print(f"No position matches found for {gene_name}")
            return None, None
    else:
        print(f"Error fetching gene details for {gene_name}: {response.status_code}")
        return None, None

def get_sequence(genome, chromosome, start, end):
    """
    Fetch sequence data for specified range.
    """
    sequence_url = f"https://api.genome.ucsc.edu/getData/sequence?genome={genome};chrom={chromosome};start={start};end={end}"
    response = requests.get(sequence_url)
    if response.ok:
        return response.json().get('dna', '').upper()  # Convert sequence to uppercase
    else:
        print(f"Error fetching sequence for {chromosome}:{start}-{end}: {response.status_code}")
        return None

def main():
    genome = "mm39"
    output_file_100bp = "upstream_sequences_100bp_LMC.fasta"
    output_file_50bp = "upstream_sequences_50bp_LMC.fasta"

    with open('LMC_countdata_allexpressed_genes.csv', newline='') as csvfile:
        gene_reader = csv.reader(csvfile)
        next(gene_reader, None)  # Skip the header row

        with open(output_file_100bp, "w") as file_100bp, open(output_file_50bp, "w") as file_50bp:
            for row in gene_reader:
                gene = row[0].upper()  # Convert gene name to uppercase

                # Skip genes starting with GM, RP, or HB-
                if gene.startswith(("GM", "RP", "HB-")):
                    continue

                chromosome, tss_position = get_gene_details(gene, genome)
                if chromosome and tss_position is not None:
                    # Fetch and write 100 bp sequence
                    sequence_100 = get_sequence(genome, chromosome, max(0, tss_position - 100), tss_position)
                    if sequence_100:
                        file_100bp.write(f">{gene}\n{sequence_100}\n")

                    # Fetch and write 50 bp sequence
                    sequence_50 = get_sequence(genome, chromosome, max(0, tss_position - 50), tss_position)
                    if sequence_50:
                        file_50bp.write(f">{gene}\n{sequence_50}\n")
                    else:
                        print(f"Failed to retrieve sequence for {gene}")
                else:
                    print(f"Failed to retrieve data for {gene}")

if __name__ == "__main__":
    main()
